# Genome-wide coverage plots with mosdepth

This notebook shows how I go from paired-end reads to a normalized, genome-wide read-depth plot. Plots like this are a quick way to spot aneuploidy, segmental duplications and deletions.

**Workflow**

1. Map reads with `minimap2`, keep reads with MAPQ ≥ 30, sort and index with `samtools`
2. Compute mean depth in 500 bp windows with `mosdepth`
3. Normalize each sample to its genome-wide median depth and plot in R (`ggplot2`)

**Tools**

- Command line: `minimap2`, `samtools`, `mosdepth`
- R: `dplyr`, `readr`, `ggplot2`
- To run the R cells inside this notebook: `pip install rpy2` (Python kernel). Alternatively, paste the R cells into RStudio.

**Placeholders to replace with your own names**

| Placeholder | Meaning |
|---|---|
| `reference_genome.fa` | Reference genome FASTA |
| `Sample1_S1`, `Sample2_S2`, … | Read/file prefixes (everything before `_R1_001.fastq.gz`) |
| `Strain_A`, `Strain_B`, … | Strain names shown on the plot |

**Citation:** If you use this notebook in your research, please cite [Huang et al. (2026), bioRxiv](https://www.biorxiv.org/content/10.64898/2026.09.15.751489v1), DOI: [10.64898/2026.09.15.751489](https://doi.org/10.64898/2026.09.15.751489).


In [ ]:
%load_ext rpy2.ipython

## Step 1 — Map reads and filter by mapping quality

Input files are paired-end reads named like `Sample1_S1_R1_001.fastq.gz` / `Sample1_S1_R2_001.fastq.gz`, in the same folder as the reference.

- `-ax sr` — minimap2 preset for short paired-end reads, SAM output
- `samtools view -h -q 30` — keeps reads with MAPQ ≥ 30, which removes reads that map equally well to several places (repeats, transposons)
- Output: `Sample1_S1_filteredq30.bam` + `.bai` index

`samtools faidx` writes `reference_genome.fa.fai`, which the plotting step uses for chromosome names and lengths.

In [ ]:
%%bash
REF=reference_genome.fa

samtools faidx "$REF"

for f in *R1*.fastq.gz
do
n="${f%_R1*}"
minimap2 \
-ax sr \
"$REF" \
"$n"_R1_001.fastq.gz \
"$n"_R2_001.fastq.gz |\
samtools view -h -q 30 |\
samtools sort -o "$n"_filteredq30.bam
samtools index "$n"_filteredq30.bam
done

## Step 2 — Window depth with mosdepth

`--by 500` reports the mean depth in consecutive 500 bp windows. For each sample the file used for plotting is:

`Sample1_S1.regions.bed.gz` — columns: chromosome, start, end, mean depth

In [ ]:
%%bash
for f in *_filteredq30.bam
do
n="${f%_filteredq30.bam}"
mosdepth "$n" "$f" \
--by 500 \
--threads 2
done

## Step 3 — Normalize and plot in R

Each window's depth is divided by that sample's **genome-wide median** depth, so:

- **1** = the sample's median coverage baseline
- **2** = twice the baseline coverage, consistent with duplication (or disomy in a predominantly haploid sample)
- **0** = no retained coverage, which may reflect deletion or poor mappability

The median is used rather than the mean because it is barely shifted by an aneuploid chromosome or by high-copy repeats.

Edit the **Settings** block at the top of the cell (sample table, colours, `.fai` path), then run it. The cell prints a per-sample summary, shows the plot, and saves it as a PDF.

To make the PDF smaller, install `ggrastr` + `ragg` and swap the `geom_point(...)` line for the commented `ggrastr::rasterise(...)` line. The points become a 600 dpi image, while axes and text stay editable in Illustrator.

In [ ]:
%%R -w 8 -h 5.5 --units in -r 150
library(dplyr)
library(readr)
library(ggplot2)

# ==========================================================
# Settings
# ==========================================================
fai_path <- "reference_genome.fa.fai"

# file_prefix = the "$n" from steps 1-2; strain = label on the plot
samples <- tibble::tribble(
  ~file_prefix,   ~strain,
  "Sample1_S1",   "Strain_A",
  "Sample2_S2",   "Strain_B",
  "Sample3_S3",   "Strain_C",
  "Sample4_S4",   "Reference"
)

colors <- c(
  "Strain_A"  = "#80b1d3",
  "Strain_B"  = "#fb8072",
  "Strain_C"  = "#fdb462",
  "Reference" = "#9e9e9e"
)

y_max       <- 2.5   # y-axis ceiling; higher windows are clipped, not removed
output_file <- "Mosdepth_normalized_coverage.pdf"

# ==========================================================
# Chromosome layout (end to end, in .fai order)
# ==========================================================
chr <- read_tsv(fai_path,
                col_names = c("chrom", "length", "offset", "linebases", "linewidth"),
                col_types = "cdddd") %>%
  select(chrom, length) %>%
  # filter(chrom != "mito") %>%   # drop contigs you don't want plotted
  mutate(chr_start = cumsum(length) - length,
         chr_mid   = chr_start + length / 2)

genome_length <- sum(chr$length)

# ==========================================================
# Read mosdepth output and normalize
# ==========================================================
read_sample <- function(prefix, strain) {
  read_tsv(paste0(prefix, ".regions.bed.gz"),
           col_names = c("chrom", "start", "end", "depth"),
           col_types = "cddd", progress = FALSE) %>%
    inner_join(select(chr, chrom, chr_start), by = "chrom") %>%   # keeps only plotted chromosomes
    mutate(pos    = chr_start + (start + end) / 2,               # genome-wide window midpoint
           norm   = depth / median(depth),
           strain = strain)
}

cov <- bind_rows(Map(read_sample, samples$file_prefix, samples$strain))
cov$strain <- factor(cov$strain, levels = samples$strain)

print(
  cov %>%
    group_by(strain) %>%
    summarise(windows       = n(),
              median_depth  = median(depth),
              mean_depth    = mean(depth),
              above_ceiling = sum(norm > y_max))
)

# ==========================================================
# Plot
# ==========================================================
label_data <- tibble(strain = factor(samples$strain, levels = samples$strain),
                     pos    = 0,
                     norm   = y_max * 0.94)

p <- ggplot(cov, aes(pos, norm, colour = strain)) +
  geom_point(size = 0.15, alpha = 0.8) +
  # ggrastr::rasterise(geom_point(size = 0.15, alpha = 0.8), dpi = 600, dev = "ragg") +
  geom_hline(yintercept = 1, linetype = "dashed", linewidth = 0.3, colour = "grey40") +
  geom_vline(xintercept = c(chr$chr_start, genome_length),
             linetype = "dashed", linewidth = 0.25, colour = "grey60") +
  geom_text(data = label_data, aes(label = strain), colour = "black",
            hjust = -0.1, vjust = 1, size = 3.5) +
  facet_grid(rows = vars(strain)) +
  scale_colour_manual(values = colors, guide = "none") +
  scale_x_continuous(limits = c(0, genome_length), breaks = chr$chr_mid,
                     labels = chr$chrom, expand = c(0, 0)) +
  scale_y_continuous(breaks = c(0, 1, 2), expand = c(0, 0)) +
  coord_cartesian(ylim = c(0, y_max)) +
  labs(x = NULL, y = "Normalized coverage") +
  theme_classic() +
  theme(strip.text       = element_blank(),
        strip.background = element_blank(),
        panel.spacing.y  = unit(3, "pt"),
        panel.border     = element_rect(colour = "black", fill = NA, linewidth = 0.25),
        axis.title.y     = element_text(size = 9),
        axis.text.y      = element_text(size = 7),
        axis.text.x      = element_text(size = 7, angle = 45, hjust = 1))

# ==========================================================
# Save + show
# ==========================================================
ggsave(output_file, p, width = 8, height = 1.1 * nrow(samples) + 1, device = pdf)
print(p)